# 17 — Stochastic Interpolant Results

Downloads evaluation artifacts from W&B for all SI variants and renders:
- Metrics tables (MAE, MP-MAE, KH-MAE, depth/area, calibration)
- Overlay plots at a fixed frame index (default: 70)
- Ensemble-mean vs GT comparison
- Per-pixel ensemble std maps
- Comparison across sigma, gamma_c, and conditioning variants

**No local data or checkpoints required.** Run `evaluate_coupled.py` on TRACE first.

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import wandb
from IPython.display import display

# ── CONFIG ─────────────────────────────────────────────────────────────────
WANDB_ENTITY  = os.getenv('WANDB_ENTITY', 'your-entity')
WANDB_PROJECT = 'Flow3D_SuperResolution'
PLOT_FRAME_IDX = 70        # frame index for overlay plots (matches --plot_frame_idx in evaluate_coupled.py)
LOCAL_DIR     = '/tmp/si_eval'

# Map of experiment run_name → display label and color
# Edit these to match the --run_name values used when running evaluate_coupled.py
EXPERIMENTS = {
    # Single-field (temperature)
    'si_opt_a_s005_ulr_temp': {'label': 'A σ=0.05 ULR (temp)',   'color': '#4c72b0', 'marker': 'o'},
    'si_opt_a_s010_ulr_temp': {'label': 'A σ=0.10 ULR (temp)',   'color': '#dd8452', 'marker': 'o'},
    'si_opt_a_s020_ulr_temp': {'label': 'A σ=0.20 ULR (temp)',   'color': '#55a868', 'marker': 'o'},
    'si_opt_a_s010_enc_temp': {'label': 'A σ=0.10 ENC (temp)',   'color': '#c44e52', 'marker': 's'},
    'si_opt_b_c010_ulr_temp': {'label': 'B c=0.10 ULR (temp)',   'color': '#8172b2', 'marker': '^'},
    'si_opt_b_c025_ulr_temp': {'label': 'B c=0.25 ULR (temp)',   'color': '#937860', 'marker': '^'},
    # Multifield (temperature + sdfliqlabel)
    'si_opt_a_s005_ulr_multi': {'label': 'A σ=0.05 ULR (multi)', 'color': '#4c72b0', 'marker': 'D'},
    'si_opt_a_s010_ulr_multi': {'label': 'A σ=0.10 ULR (multi)', 'color': '#dd8452', 'marker': 'D'},
    'si_opt_a_s020_ulr_multi': {'label': 'A σ=0.20 ULR (multi)', 'color': '#55a868', 'marker': 'D'},
    'si_opt_a_s010_enc_multi': {'label': 'A σ=0.10 ENC (multi)', 'color': '#c44e52', 'marker': 'P'},
    'si_opt_b_c010_ulr_multi': {'label': 'B c=0.10 ULR (multi)', 'color': '#8172b2', 'marker': 'v'},
    'si_opt_b_c025_ulr_multi': {'label': 'B c=0.25 ULR (multi)', 'color': '#937860', 'marker': 'v'},
}
# ───────────────────────────────────────────────────────────────────────────

os.makedirs(LOCAL_DIR, exist_ok=True)
api = wandb.Api()

In [ ]:
# Download artifacts and load results
data = {}
for run_name in EXPERIMENTS:
    art_name = f'{WANDB_ENTITY}/{WANDB_PROJECT}/{run_name}_eval_predictions:latest'
    dl_dir = os.path.join(LOCAL_DIR, run_name)
    try:
        art = api.artifact(art_name)
        art.download(root=dl_dir)
        # Load JSON results
        json_path = os.path.join(dl_dir, 'results.json')
        if os.path.exists(json_path):
            with open(json_path) as f:
                results = json.load(f)
        else:
            results = {}
        # Load npz
        npz_path = os.path.join(dl_dir, 'predictions.npz')
        npz = np.load(npz_path, allow_pickle=True) if os.path.exists(npz_path) else None
        data[run_name] = {'results': results, 'npz': npz, 'dl_dir': dl_dir}
        print(f'[OK] {run_name}')
    except Exception as e:
        print(f'[MISSING] {run_name}: {e}')
        data[run_name] = {'results': {}, 'npz': None, 'dl_dir': dl_dir}

## Metrics Table — Single Field

In [ ]:
def make_metrics_table(run_names):
    rows = []
    for rn in run_names:
        r = data.get(rn, {}).get('results', {})
        if not r:
            continue
        rows.append({
            'Experiment':        EXPERIMENTS[rn]['label'],
            'MAE (K)':           f"{r.get('MAE_mean', float('nan')):.1f} ± {r.get('MAE_std', 0):.1f}",
            'Ens-MAE (K)':       f"{r.get('MAE_ensemble_mean', float('nan')):.1f}",
            'RMSE (K)':          f"{r.get('RMSE_mean', float('nan')):.1f}",
            'MP-MAE (µm)':       f"{r.get('MP_MAE_mean', float('nan')):.1f}",
            'KH-MAE (µm)':       f"{r.get('KH_MAE_mean', float('nan')):.1f}",
            'Depth MAE (µm)':    f"{r.get('depth_mae_um', float('nan')):.1f}",
            'Area MAE (µm²)':    f"{r.get('area_mae_um2', float('nan')):.0f}",
            'Calib 2σ':          f"{r.get('calibration_2sigma', float('nan')):.2f}",
        })
    return pd.DataFrame(rows)

temp_runs  = [k for k in EXPERIMENTS if k.endswith('_temp')  and k in data]
multi_runs = [k for k in EXPERIMENTS if k.endswith('_multi') and k in data]

print('=== Single Field (temperature) ===')
display(make_metrics_table(temp_runs))

## Metrics Table — Multifield

In [ ]:
print('=== Multifield (temperature + sdfliqlabel) ===')
display(make_metrics_table(multi_runs))

## Sigma Sweep — Option A (MAE vs σ)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
for ax, tag, title in zip(axes, ['temp', 'multi'], ['Single field', 'Multifield']):
    sigmas, maes, mp_maes = [], [], []
    for s in [0.05, 0.10, 0.20]:
        rn = f'si_opt_a_s{int(s*100):03d}_ulr_{tag}'
        r = data.get(rn, {}).get('results', {})
        if r:
            sigmas.append(s)
            maes.append(r.get('MAE_mean', float('nan')))
            mp_maes.append(r.get('MP_MAE_mean', float('nan')))
    ax.plot(sigmas, maes,    'o-', label='MAE (K)',     color='#4c72b0')
    ax2 = ax.twinx()
    ax2.plot(sigmas, mp_maes,'s--',label='MP-MAE (µm)',color='#dd8452')
    ax.set_xlabel('σ'); ax.set_ylabel('MAE (K)', color='#4c72b0')
    ax2.set_ylabel('MP-MAE (µm)', color='#dd8452')
    ax.set_title(f'Option A sigma sweep — {title}')
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labs1+labs2, fontsize=8)
plt.tight_layout(); plt.show()

## Overlay Plots at Frame Index 70 — Single Field

In [ ]:
def show_overlay(run_name, vmin=293, vmax=5000, temp_ch=0):
    """Show overlay PNG saved by evaluate_coupled.py at the configured frame index."""
    entry = data.get(run_name, {})
    dl_dir = entry.get('dl_dir', '')
    overlay = os.path.join(dl_dir, 'overlay.png')
    if not os.path.exists(overlay):
        print(f'  No overlay found for {run_name}')
        return
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.imshow(mpimg.imread(overlay))
    ax.axis('off')
    ax.set_title(f'{EXPERIMENTS[run_name]["label"]} — frame {PLOT_FRAME_IDX}', fontsize=10)
    plt.tight_layout(); plt.show()

for rn in temp_runs:
    show_overlay(rn)

## Overlay Plots at Frame Index 70 — Multifield

In [ ]:
for rn in multi_runs:
    show_overlay(rn)

## Ensemble Std Maps at Frame 70 (from npz)

In [ ]:
def plot_std_maps(run_names, frame_idx=PLOT_FRAME_IDX, temp_ch=0):
    n = sum(1 for rn in run_names if data.get(rn, {}).get('npz') is not None)
    if n == 0:
        print('No npz data available.'); return
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4), dpi=120)
    if n == 1: axes = [axes]
    ax_i = 0
    for rn in run_names:
        npz = data.get(rn, {}).get('npz')
        if npz is None: continue
        ens_std = npz['ens_std']   # (N, C, H, W)
        idx = min(frame_idx, ens_std.shape[0]-1)
        im = axes[ax_i].imshow(ens_std[idx, temp_ch].T, origin='lower', cmap='hot', vmin=0)
        axes[ax_i].set_title(EXPERIMENTS[rn]['label'], fontsize=8); axes[ax_i].axis('off')
        plt.colorbar(im, ax=axes[ax_i], fraction=0.046, label='std (K)')
        ax_i += 1
    plt.suptitle(f'Ensemble std at frame {frame_idx}', fontsize=10)
    plt.tight_layout(); plt.show()

print('Single field:')
plot_std_maps(temp_runs)
print('Multifield:')
plot_std_maps(multi_runs)

## Bar Chart — MAE Comparison (all variants)

In [ ]:
def bar_comparison(run_names, metric_key, ylabel, title):
    labels = [EXPERIMENTS[rn]['label'] for rn in run_names if data.get(rn, {}).get('results')]
    values = [data[rn]['results'].get(metric_key, float('nan'))
              for rn in run_names if data.get(rn, {}).get('results')]
    colors = [EXPERIMENTS[rn]['color'] for rn in run_names if data.get(rn, {}).get('results')]
    if not labels: print(f'No data for {title}'); return
    fig, ax = plt.subplots(figsize=(max(8, len(labels)*1.3), 4), dpi=120)
    bars = ax.bar(range(len(labels)), values, color=colors, edgecolor='w', alpha=0.9)
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(title)
    for bar, val in zip(bars, values):
        if not np.isnan(val):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                    f'{val:.1f}', ha='center', fontsize=7)
    plt.tight_layout(); plt.show()

bar_comparison(temp_runs,  'MAE_mean',            'MAE (K)',    'MAE — single field variants')
bar_comparison(multi_runs, 'MAE_mean',            'MAE (K)',    'MAE — multifield variants')
bar_comparison(temp_runs,  'MP_MAE_mean',         'MP-MAE (µm)','MP-MAE — single field variants')
bar_comparison(multi_runs, 'MP_MAE_mean',         'MP-MAE (µm)','MP-MAE — multifield variants')
bar_comparison(temp_runs,  'calibration_2sigma',  'Fraction',   'Calibration 2σ — single field')
bar_comparison(multi_runs, 'calibration_2sigma',  'Fraction',   'Calibration 2σ — multifield')

## Option B: eps0 Sweep

`evaluate_coupled.py --sampler em --eps0 <value>` must be run separately for each eps0 value.
Set `EPS0_RUNS` below to the run_names used for those sweeps.

In [ ]:
# Example: set these to the actual run_names from the eps0 sweep
EPS0_RUNS = {
    # 'eps0=0.00': 'si_opt_b_c025_ulr_temp_eps000',
    # 'eps0=0.10': 'si_opt_b_c025_ulr_temp_eps010',
    # 'eps0=0.25': 'si_opt_b_c025_ulr_temp_eps025',
    # 'eps0=0.50': 'si_opt_b_c025_ulr_temp_eps050',
    # 'eps0=1.00': 'si_opt_b_c025_ulr_temp_eps100',
}
eps0_vals = []; mae_vals = []; mp_vals = []
for label, rn in EPS0_RUNS.items():
    r = data.get(rn, {}).get('results', {})
    if r:
        eps0_vals.append(float(label.split('=')[1]))
        mae_vals.append(r.get('MAE_mean', float('nan')))
        mp_vals.append(r.get('MP_MAE_mean', float('nan')))

if eps0_vals:
    fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
    ax.plot(eps0_vals, mae_vals, 'o-', label='MAE (K)', color='#4c72b0')
    ax2 = ax.twinx()
    ax2.plot(eps0_vals, mp_vals, 's--', label='MP-MAE (µm)', color='#dd8452')
    ax.set_xlabel('eps0'); ax.set_ylabel('MAE (K)')
    ax2.set_ylabel('MP-MAE (µm)')
    ax.set_title('Option B: eps0 sweep')
    lines1, labs1 = ax.get_legend_handles_labels()
    lines2, labs2 = ax2.get_legend_handles_labels()
    ax.legend(lines1+lines2, labs1+labs2, fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('No eps0 sweep data yet — run evaluate_coupled.py with --sampler em --eps0 <value>')

## Raw Results JSON

In [ ]:
for rn, entry in data.items():
    r = entry.get('results', {})
    if r:
        print(f'\n=== {EXPERIMENTS[rn]["label"]} ===')
        print(json.dumps(r, indent=2))